# Round-0 loss generator lab

Goal: isolate the generative model from the active-learning loop. We load a fixed round-0 pool with surrogate losses, train a DDPM or Flow Matching model on `(state, params, loss)`, then score generated states by advancing them one PDE step and measuring the frozen round-0 surrogate error.


In [ ]:
from pathlib import Path
import json
import numpy as np

RUN_DIR = Path('/bettik/PROJECTS/pr-melissa/cesarpi-ext/poolbased_surrogate_runs/ks800_flow_variants/tail_paramcond_seed101')
LAB_DIR = Path('/bettik/PROJECTS/pr-melissa/cesarpi-ext/poolbased_surrogate_runs/loss_generator_lab/tail_paramcond_seed101')
LAB_DIR.mkdir(parents=True, exist_ok=True)
DATASET = LAB_DIR / 'round0_loss_dataset.npz'
GEN_DIR = LAB_DIR / 'flow_matching_conditioned'


In [ ]:
from poolbased_surrogate.loss_generator_lab import build_loss_dataset_from_run

dataset_path = build_loss_dataset_from_run(RUN_DIR, DATASET, round_id=0, split_seed=101)
dataset_path

In [ ]:
with np.load(DATASET, allow_pickle=False) as data:
    print(data.files)
    meta = json.loads(str(data['metadata']))
    print(json.dumps(meta, indent=2))
    losses = data['losses']
    loss_norm = data['loss_norm']
    split = data['split']
    print('loss quantiles:', np.quantile(losses, [0, .5, .9, .95, .99, 1]))
    print('norm loss quantiles:', np.quantile(loss_norm, [0, .5, .9, .95, .99, 1]))
    print('split counts:', {int(k): int((split == k).sum()) for k in np.unique(split)})

Train a first fast Flow Matching model. Start small here; after we see conditioning correlation and diversity, increase epochs/hidden/steps only if needed.

In [ ]:
from poolbased_surrogate.loss_generator_lab import train_loss_generator

ckpt, last = train_loss_generator(
    dataset_path=DATASET,
    output_dir=GEN_DIR,
    generator='flow_matching',
    param_mode='condition',
    epochs=40,
    batch_size=256,
    lr=2e-4,
    hidden=128,
    steps=16,
    residual_blocks=2,
    kernel_size=7,
    loss_condition_scale=1.0,
    seed=101,
    device_name='auto',
)
ckpt, last

Score generated samples with the frozen round-0 surrogate. This is the key test: if target loss and realized surrogate difficulty are uncorrelated, the generator block is not ready to be plugged back into active learning.

In [ ]:
from poolbased_surrogate.loss_generator_lab import score_generated_samples

metrics = score_generated_samples(
    checkpoint_path=ckpt,
    dataset_path=DATASET,
    output_dir=GEN_DIR,
    n_samples=1024,
    seed=101,
    device_name='auto',
    solver_batch_size=512,
)
print(json.dumps(metrics, indent=2))

In [ ]:
import matplotlib.pyplot as plt

with np.load(GEN_DIR / 'generated_scored_samples.npz') as gen, np.load(DATASET) as data:
    realized = gen['realized_losses']
    target = gen['target_loss_norm']
    ref = gen['reference_losses']
    states = gen['states'][:16, 0]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(ref, bins=60, alpha=.6, label='round0 holdout')
axes[0].hist(realized, bins=60, alpha=.6, label='generated')
axes[0].set_title('Surrogate loss distribution')
axes[0].legend()

axes[1].scatter(target, realized, s=6, alpha=.35)
axes[1].set_xlabel('target normalized loss')
axes[1].set_ylabel('realized surrogate MSE')
axes[1].set_title('Conditioning calibration')

im = axes[2].imshow(states, aspect='auto', cmap='coolwarm')
axes[2].set_title('Generated states')
fig.colorbar(im, ax=axes[2], shrink=.8)
plt.tight_layout()

Useful CLI equivalents:

```bash
python scripts/build_loss_generator_dataset.py --run-dir /path/to/run --output /path/to/round0_loss_dataset.npz
python scripts/train_loss_generator.py --dataset /path/to/round0_loss_dataset.npz --output-dir /path/to/generator --generator flow_matching --epochs 40 --score-samples 1024
```
